# 03 - Data Cleaning

## Objective

This notebook applies documented and reproducible cleaning rules to the
Online Retail transaction dataset before customer-level feature engineering.

The cleaning process is designed to:

- retain only records that can support reliable customer segmentation;
- prevent duplicate or invalid transactions from distorting customer behaviour;
- preserve business-significant activity such as cancellations and returns;
- produce a clean transaction dataset for downstream feature engineering;
- maintain a clear audit trail between the source dataset and cleaned output.

Cleaning decisions are based on the findings identified during the
data-understanding and profiling stage.

In [1]:
# necessary libraries
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import Data
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import AccountKeyConfiguration
import mltable
from mltable import MLTableHeaders, MLTableFileEncoding

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

# Load the registered Azure ML Data Asset so that analysis is based on the
# governed MLTable rather than a local file path.
retail_asset =ml_client.data.get(
    name="online-retail-mltable",
    version="1"
)

# Load the MLTable definition and materialise the dataset as a Pandas DataFrame
# for interactive profiling and exploratory analysis.
retail_table = mltable.load(retail_asset.path)
df = retail_table.to_pandas_dataframe()

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [2]:
# Confirm that the expected source dataset has been loaded.

print("Source rows:", f"{len(df):,}")
print("Source columns:", df.shape[1])

df.head()

Source rows: 541,909
Source columns: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
# Create a working copy for cleaning so that the original DataFrame remains
# available for validation and reconciliation throughout this notebook.

df_clean = df.copy()

print("Working dataset shape:", df_clean.shape)

Working dataset shape: (541909, 8)


## 2. Establish the pre-cleaning baseline

A baseline is recorded before any records are removed so that the impact of
each cleaning rule can be quantified and reconciled.

In [4]:
# Capture baseline metrics before applying any cleaning rules.

baseline_rows = len(df_clean)
baseline_customers = df_clean["CustomerID"].nunique()

print(f"Source rows: {baseline_rows:,}")
print(f"Identifiable customers: {baseline_customers:,}")

Source rows: 541,909
Identifiable customers: 4,372


## 3. Customer identification

Customer-level segmentation requires transactions to be attributable to a
known customer.

Records without a `CustomerID` cannot be reliably aggregated to an individual
customer and therefore cannot contribute directly to the clustering dataset.

In [5]:
# Identify transactions that cannot be attributed to a known customer.

missing_customer_mask = df_clean["CustomerID"].isna()

missing_customer_count = missing_customer_mask.sum()
missing_customer_percentage = (
    missing_customer_count / len(df_clean) * 100
)

print(f"Rows with missing CustomerID: {missing_customer_count:,}")
print(f"Percentage of rows: {missing_customer_percentage:.2f}%")

Rows with missing CustomerID: 135,080
Percentage of rows: 24.93%


In [6]:
# Review a sample of transactions without CustomerID before applying
# the exclusion rule.

df_clean.loc[missing_customer_mask].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,<NA>,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,NaN,United Kingdom
1447,536544,21790,VINTAGE SNAP CARDS,9,2010-12-01 14:32:00,1.66,NaN,United Kingdom
1448,536544,21791,VINTAGE HEADS AND TAILS CARD GAME,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1449,536544,21801,CHRISTMAS TREE DECORATION WITH BELL,10,2010-12-01 14:32:00,0.43,NaN,United Kingdom
1450,536544,21802,CHRISTMAS TREE HEART DECORATION,9,2010-12-01 14:32:00,0.43,NaN,United Kingdom
1451,536544,21803,CHRISTMAS TREE STAR DECORATION,11,2010-12-01 14:32:00,0.43,NaN,United Kingdom


In [7]:
# Exclude transactions that cannot be assigned to an identifiable customer.

rows_before = len(df_clean)

df_clean = df_clean.loc[
    df_clean["CustomerID"].notna()
].copy()

rows_removed = rows_before - len(df_clean)

print(f"Rows removed: {rows_removed:,}")
print(f"Rows remaining: {len(df_clean):,}")

Rows removed: 135,080
Rows remaining: 406,829


## 4. Exact Duplicate Transaction Lines

Records were assessed for exact duplication across all available transaction
fields.

Rows are considered probable duplicates only when all source attributes are
identical, including invoice number, product, quantity, transaction timestamp,
unit price, customer and country.

Because the dataset does not contain a unique invoice-line identifier, exact
duplicates cannot be independently verified against the source system.
However, identical transaction lines within the same invoice are treated as
duplicate source records for this analysis.

One copy of each exact duplicate is retained.

In [8]:
# Identify all rows that have an identical match across every source column.
duplicate_mask = df_clean.duplicated(keep=False)

# Keep only repeated rows and sort them so matching records appear together.
duplicate_rows = (
    df_clean.loc[duplicate_mask]
    .sort_values(
        by=[
            "InvoiceNo",
            "CustomerID",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "Country"
        ]
    )
)

print(f"Rows involved in exact duplicate groups: {len(duplicate_rows):,}")
duplicate_rows.head(30)

Rows involved in exact duplicate groups: 10,062


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom
598,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom


In [9]:
duplicate_rows = (
    df_clean.loc[duplicate_mask]
    .copy()
)

duplicate_rows["repeat_count"] = (
    duplicate_rows
    .groupby(
        [
            "InvoiceNo",
            "CustomerID",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "Country"
        ]
    )["InvoiceNo"]
    .transform("size")
)

duplicate_rows = duplicate_rows.sort_values(
    by=[
        "InvoiceNo",
        "CustomerID",
        "StockCode",
        "Description"
    ]
)

duplicate_rows.head(30)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,repeat_count
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom,2
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom,2
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom,2
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom,2
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom,2
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom,2
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom,2
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom,2
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920.0,United Kingdom,3
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom,2


In [10]:
# Count only the additional copies.
# The first occurrence of each transaction line will be retained.

duplicates_to_remove = df_clean.duplicated(keep="first").sum()

print(f"Duplicate rows to remove: {duplicates_to_remove:,}")

Duplicate rows to remove: 5,225


In [11]:
# Remove repeated copies of exact transaction lines while retaining
# the first occurrence of each transaction.

rows_before = len(df_clean)

df_clean = (
    df_clean
    .drop_duplicates(keep="first")
    .copy()
)

rows_removed = rows_before - len(df_clean)

print(f"Duplicate rows removed: {rows_removed:,}")
print(f"Rows remaining: {len(df_clean):,}")

Duplicate rows removed: 5,225
Rows remaining: 401,604


In [12]:
duplicate_removal_summary = {
    "rule": "Exact duplicate transaction lines",
    "rows_before": rows_before,
    "rows_removed": rows_removed,
    "rows_after": len(df_clean)
}

duplicate_removal_summary

{'rule': 'Exact duplicate transaction lines',
 'rows_before': 406829,
 'rows_removed': 5225,
 'rows_after': 401604}

## 5. Cancelled Invoices and Negative Quantities

Cancelled invoices and negative quantities may represent returns, reversals
or other non-standard transaction activity.

Before deciding how these records should be treated, this section assesses
the relationship between:

- invoice numbers beginning with `C`; and
- transactions where `Quantity < 0`.

The objective is to determine whether these conditions describe the same
business event and whether they should be excluded from positive-purchase
features or retained separately as behavioural indicators.

In [13]:
# Identify cancelled invoices using the source-system convention that
# cancelled InvoiceNo values begin with the letter "C".

cancelled_mask = (
    df_clean["InvoiceNo"]
    .astype("string")
    .str.startswith("C", na=False)
)

# Identify transactions where the recorded quantity is negative.

negative_quantity_mask = df_clean["Quantity"] < 0

# Count cancelled invoice rows and negative-quantity rows separately.

cancelled_count = cancelled_mask.sum()
negative_quantity_count = negative_quantity_mask.sum()

print(f"Cancelled invoice rows: {cancelled_count:,}")
print(f"Negative quantity rows: {negative_quantity_count:,}")

Cancelled invoice rows: 8,872
Negative quantity rows: 8,872


In [14]:
# Measure how many transactions meet both conditions.

cancelled_and_negative = (
    cancelled_mask & negative_quantity_mask
).sum()

print(
    f"Rows that are both cancelled and negative quantity: "
    f"{cancelled_and_negative:,}")

Rows that are both cancelled and negative quantity: 8,872


In [15]:
# Identify negative quantities whose InvoiceNo does not begin with "C".

negative_not_cancelled = df_clean.loc[
    negative_quantity_mask & ~cancelled_mask
]

print(
    "Negative quantities not marked as cancelled:",
    len(negative_not_cancelled)
)

Negative quantities not marked as cancelled: 0


In [16]:
# Identify cancelled invoice rows that do not have a negative quantity.

cancelled_not_negative = df_clean.loc[
    cancelled_mask & ~negative_quantity_mask
]

print(
    "Cancelled invoices without negative quantity:",
    len(cancelled_not_negative)
)

Cancelled invoices without negative quantity: 0


In [17]:
# Summarise the relationship between cancellation status and quantity sign.
import pandas as pd
cancellation_summary = pd.DataFrame({
    "Measure": [
        "Cancelled invoice rows",
        "Negative quantity rows",
        "Cancelled AND negative quantity",
        "Cancelled but not negative",
        "Negative but not cancelled"
    ],
    "Count": [
        cancelled_count,
        negative_quantity_count,
        cancelled_and_negative,
        len(cancelled_not_negative),
        len(negative_not_cancelled)
    ]
})

cancellation_summary

,Measure,Count
0,Cancelled invoice rows,8872
1,Negative quantity rows,8872
2,Cancelled AND negative quantity,8872
3,Cancelled but not negative,0
4,Negative but not cancelled,0


### Finding

Cancelled invoices and negative quantities are fully aligned in the current
dataset.

All 8,872 transactions with an invoice number beginning with `C` also have a
negative quantity, and no negative-quantity transactions exist outside
cancelled invoices.

This indicates that the two indicators represent the same cancellation/return
activity in this dataset.

Cancelled transactions will therefore be excluded from positive purchase
calculations. However, they will be retained separately so that cancellation
or return behaviour can be considered during customer-level feature
engineering.

In [18]:
# Preserve cancelled/returned transactions separately so that return behaviour
# can be incorporated into customer-level features later if it proves useful.

df_returns = df_clean.loc[
    cancelled_mask
].copy()

# Create the positive-purchase dataset used for standard purchasing behaviour.
# Because cancelled invoices and negative quantities are fully aligned,
# filtering on the cancellation flag is sufficient.

df_purchases = df_clean.loc[
    ~cancelled_mask
].copy()

print(f"Purchase transactions: {len(df_purchases):,}")
print(f"Cancelled/returned transactions: {len(df_returns):,}")
print(f"Combined transactions: {len(df_purchases) + len(df_returns):,}")

Purchase transactions: 392,732
Cancelled/returned transactions: 8,872
Combined transactions: 401,604


## 6. Assess non-positive UnitPrice values

After separating cancelled transactions, positive purchase records are
assessed for zero or negative unit prices.

Transactions with non-positive prices may represent free items, adjustments,
administrative records or data-quality issues and could distort customer
monetary-value features.

In [19]:
# Identify positive purchase transactions with a zero or negative unit price.

non_positive_price_mask = df_purchases["UnitPrice"] <= 0

non_positive_price_count = non_positive_price_mask.sum()

print(
    f"Purchase transactions with UnitPrice <= 0: "
    f"{non_positive_price_count:,}"
)

Purchase transactions with UnitPrice <= 0: 40


In [20]:
# Inspect the affected records before deciding on an appropriate treatment.

non_positive_price_rows = (
    df_purchases.loc[non_positive_price_mask]
    .sort_values(["UnitPrice", "InvoiceNo"])
)

non_positive_price_rows.head(40)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.0,12647.0,Germany
33576,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,2010-12-16 14:36:00,0.0,16560.0,United Kingdom
40089,539722,22423,REGENCY CAKESTAND 3 TIER,10,2010-12-21 13:45:00,0.0,14911.0,EIRE
47068,540372,22090,PAPER BUNTING RETROSPOT,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
47070,540372,22553,PLASTERS IN TIN SKULLS,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
56674,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,2011-01-13 15:10:00,0.0,15107.0,United Kingdom
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2011-02-10 13:08:00,0.0,17560.0,United Kingdom
130188,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,2011-03-23 10:25:00,0.0,13239.0,United Kingdom
139453,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,2011-03-30 12:45:00,0.0,13113.0,United Kingdom
145208,548871,22162,HEART GARLAND RUSTIC PADDED,2,2011-04-04 14:42:00,0.0,14410.0,United Kingdom


In [21]:
# Count zero-price and negative-price purchase transactions separately.

zero_price_count = (df_purchases["UnitPrice"] == 0).sum()
negative_price_count = (df_purchases["UnitPrice"] < 0).sum()

print(f"Zero-price transactions: {zero_price_count:,}")
print(f"Negative-price transactions: {negative_price_count:,}")

Zero-price transactions: 40
Negative-price transactions: 0


In [22]:
# Review the affected records in enough detail to understand their business meaning.

non_positive_price_rows[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country"
    ]
].sort_values(
    ["UnitPrice", "InvoiceNo", "StockCode"]
)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.0,12647.0,Germany
33576,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,2010-12-16 14:36:00,0.0,16560.0,United Kingdom
40089,539722,22423,REGENCY CAKESTAND 3 TIER,10,2010-12-21 13:45:00,0.0,14911.0,EIRE
47068,540372,22090,PAPER BUNTING RETROSPOT,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
47070,540372,22553,PLASTERS IN TIN SKULLS,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
56674,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,2011-01-13 15:10:00,0.0,15107.0,United Kingdom
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2011-02-10 13:08:00,0.0,17560.0,United Kingdom
130188,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,2011-03-23 10:25:00,0.0,13239.0,United Kingdom
139453,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,2011-03-30 12:45:00,0.0,13113.0,United Kingdom
145208,548871,22162,HEART GARLAND RUSTIC PADDED,2,2011-04-04 14:42:00,0.0,14410.0,United Kingdom


In [23]:
# Review the zero-price transactions in full so that their business meaning
# can be assessed before any cleaning rule is applied.

zero_price_rows = (
    df_purchases.loc[
        df_purchases["UnitPrice"] == 0,
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "CustomerID",
            "Country"
        ]
    ]
    .sort_values(
        ["StockCode", "Description", "InvoiceNo"]
    )
)

display(zero_price_rows)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
420404,572893,21208,PASTEL COLOUR HONEYCOMB FAN,5,2011-10-26 14:36:00,0.0,18059.0,United Kingdom
314748,564651,21786,POLKADOT RAIN HAT,144,2011-08-26 14:19:00,0.0,14646.0,Netherlands
139453,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,2011-03-30 12:45:00,0.0,13113.0,United Kingdom
130188,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,2011-03-23 10:25:00,0.0,13239.0,United Kingdom
436597,574175,22065,CHRISTMAS PUDDING TRINKET POT,12,2011-11-03 11:47:00,0.0,14110.0,United Kingdom
454464,575579,22089,PAPER BUNTING VINTAGE PAISLEY,24,2011-11-10 11:49:00,0.0,13081.0,United Kingdom
47068,540372,22090,PAPER BUNTING RETROSPOT,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
145208,548871,22162,HEART GARLAND RUSTIC PADDED,2,2011-04-04 14:42:00,0.0,14410.0,United Kingdom
279324,561284,22167,OVAL WALL MIRROR DIAMANTE,1,2011-07-26 12:24:00,0.0,16818.0,United Kingdom
56674,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,2011-01-13 15:10:00,0.0,15107.0,United Kingdom


In [24]:
# Identify whether zero-price records are concentrated in particular
# products or transaction types.

zero_price_summary = (
    zero_price_rows
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

zero_price_summary

,StockCode,Description,row_count
33,M,Manual,6
1,21786,POLKADOT RAIN HAT,1
0,21208,PASTEL COLOUR HONEYCOMB FAN,1
2,22055,MINI CAKE STAND HANGING STRAWBERY,1
3,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,1
5,22089,PAPER BUNTING VINTAGE PAISLEY,1
4,22065,CHRISTMAS PUDDING TRINKET POT,1
7,22162,HEART GARLAND RUSTIC PADDED,1
8,22167,OVAL WALL MIRROR DIAMANTE,1
9,22168,ORGANISER WOOD ANTIQUE WHITE,1


In [25]:
# Check whether zero-price items occur alongside normally priced items
# within the same invoice.

zero_price_invoices = zero_price_rows["InvoiceNo"].unique()

invoice_context = (
    df_purchases[
        df_purchases["InvoiceNo"].isin(zero_price_invoices)
    ]
    .sort_values(["InvoiceNo", "StockCode"])
)

display(invoice_context)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
9293,537197,22199,FRYING PAN RED RETROSPOT,4,2010-12-05 14:02:00,4.25,12647.0,Germany
9294,537197,22200,FRYING PAN PINK POLKADOT,4,2010-12-05 14:02:00,4.25,12647.0,Germany
9295,537197,22505,MEMO BOARD COTTAGE DESIGN,4,2010-12-05 14:02:00,4.95,12647.0,Germany
9296,537197,22507,MEMO BOARD RETROSPOT DESIGN,4,2010-12-05 14:02:00,4.95,12647.0,Germany
9301,537197,22624,IVORY KITCHEN SCALES,2,2010-12-05 14:02:00,8.50,12647.0,Germany
...,...,...,...,...,...,...,...,...
485983,577696,85099C,JUMBO BAG BAROQUE BLACK WHITE,2,2011-11-21 11:57:00,2.08,16406.0,United Kingdom
486016,577696,85123A,WHITE HANGING HEART T-LIGHT HOLDER,1,2011-11-21 11:57:00,2.95,16406.0,United Kingdom
486019,577696,85136A,YELLOW SHARK HELICOPTER,1,2011-11-21 11:57:00,7.95,16406.0,United Kingdom
485985,577696,M,Manual,1,2011-11-21 11:57:00,0.00,16406.0,United Kingdom


In [26]:
# Identify whether non-positive prices are concentrated in particular products
# or administrative transaction types.

non_positive_price_rows[
    ["StockCode", "Description"]
].value_counts().head(20)

StockCode  Description                        
M          Manual                                 6
21786      POLKADOT RAIN HAT                      1
21208      PASTEL COLOUR HONEYCOMB FAN            1
22055      MINI CAKE STAND  HANGING STRAWBERY     1
22062      CERAMIC BOWL WITH LOVE HEART DESIGN    1
22089      PAPER BUNTING VINTAGE PAISLEY          1
22065      CHRISTMAS PUDDING TRINKET POT          1
22162      HEART GARLAND RUSTIC PADDED            1
22167       OVAL WALL MIRROR DIAMANTE             1
22168      ORGANISER WOOD ANTIQUE WHITE           1
22090      PAPER BUNTING RETROSPOT                1
22423      REGENCY CAKESTAND 3 TIER               1
22437      SET OF 9 BLACK SKULL BALLOONS          1
22464      HANGING METAL HEART LANTERN            1
22553      PLASTERS IN TIN SKULLS                 1
22580      ADVENT CALENDAR GINGHAM SACK           1
22619      SET OF 6 SOLDIER SKITTLES              1
22625      RED KITCHEN SCALES                     1
22385      JUMBO 

In [27]:
# Review the range of quantities associated with non-positive prices.

non_positive_price_rows["Quantity"].describe()

count       40.000000
mean       347.100000
std       1978.311813
min          1.000000
25%          1.000000
50%          4.500000
75%         24.000000
max      12540.000000
Name: Quantity, dtype: float64

In [28]:
# Calculate line value so that the overall value of each affected invoice
# can be assessed.

invoice_context = df_purchases[
    df_purchases["InvoiceNo"].isin(
        zero_price_rows["InvoiceNo"].unique()
    )
].copy()

invoice_context["LineValue"] = (
    invoice_context["Quantity"] *
    invoice_context["UnitPrice"]
)

zero_price_invoice_summary = (
    invoice_context
    .groupby("InvoiceNo")
    .agg(
        total_lines=("StockCode", "size"),
        zero_price_lines=("UnitPrice", lambda x: (x == 0).sum()),
        positive_price_lines=("UnitPrice", lambda x: (x > 0).sum()),
        invoice_value=("LineValue", "sum")
    )
    .reset_index()
)

zero_price_invoice_summary

,InvoiceNo,total_lines,zero_price_lines,positive_price_lines,invoice_value
0,537197,17,1,16,286.50
1,539263,21,1,20,332.18
2,539722,29,1,28,762.30
3,540372,171,2,169,3527.64
4,541109,2,1,1,14.95
5,543599,1,1,0,0.00
6,547417,22,1,21,329.56
7,548318,7,1,6,453.05
8,548871,7,1,6,91.48
9,550188,23,1,22,669.25


In [29]:
# Count how many affected invoices contain at least one paid item
# and how many consist entirely of zero-price lines.
import pandas as pd
import numpy as np
zero_price_invoice_summary["invoice_type"] = np.where(
    zero_price_invoice_summary["positive_price_lines"] > 0,
    "Contains paid items",
    "Zero-price only"
)

zero_price_invoice_summary["invoice_type"].value_counts()

invoice_type
Contains paid items    30
Zero-price only         4
Name: count, dtype: int64

In [30]:
zero_price_invoice_summary["invoice_type"].value_counts()

invoice_type
Contains paid items    30
Zero-price only         4
Name: count, dtype: int64

### Zero-price invoice investigation

Most zero-price transactions occur within otherwise paid invoices, suggesting
that these lines may represent free items, promotional products or other
non-billed items associated with genuine customer purchases.

Four invoices contain only zero-price transaction lines. These invoices require
further investigation because they do not represent a measurable monetary
purchase and could affect frequency-based customer features if treated as
standard orders.

In [31]:
# Identify invoice numbers where every transaction line has UnitPrice = 0.

zero_only_invoice_numbers = (
    zero_price_invoice_summary.loc[
        zero_price_invoice_summary["invoice_type"] == "Zero-price only",
        "InvoiceNo"
    ]
)

zero_only_invoices = (
    df_purchases[
        df_purchases["InvoiceNo"].isin(zero_only_invoice_numbers)
    ]
    .sort_values(
        ["InvoiceNo", "StockCode"]
    )
)

display(zero_only_invoices)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2011-02-10 13:08:00,0.0,17560.0,United Kingdom
314748,564651,21786,POLKADOT RAIN HAT,144,2011-08-26 14:19:00,0.0,14646.0,Netherlands
314747,564651,22955,36 FOIL STAR CAKE CASES,144,2011-08-26 14:19:00,0.0,14646.0,Netherlands
314746,564651,23268,SET OF 2 CERAMIC CHRISTMAS REINDEER,192,2011-08-26 14:19:00,0.0,14646.0,Netherlands
314745,564651,23270,SET OF 2 CERAMIC PAINTED HEARTS,96,2011-08-26 14:19:00,0.0,14646.0,Netherlands
361825,568384,M,Manual,1,2011-09-27 09:46:00,0.0,12748.0,United Kingdom
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.0,13256.0,United Kingdom


In [32]:
# Summarise the zero-price-only invoices to understand their size,
# customer ownership and product composition.

zero_only_summary = (
    zero_only_invoices
    .groupby(
        ["InvoiceNo", "CustomerID"],
        dropna=False
    )
    .agg(
        transaction_lines=("StockCode", "size"),
        total_quantity=("Quantity", "sum"),
        unique_products=("StockCode", "nunique"),
        invoice_value=("UnitPrice", lambda x: 0)
    )
    .reset_index()
)

zero_only_summary

,InvoiceNo,CustomerID,transaction_lines,total_quantity,unique_products,invoice_value
0,543599,17560.0,1,16,1,0
1,564651,14646.0,4,576,4,0
2,568384,12748.0,1,1,1,0
3,578841,13256.0,1,12540,1,0


In [33]:
# Review the products and descriptions associated with zero-price-only invoices.

zero_only_invoices[
    [
        "InvoiceNo",
        "CustomerID",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "InvoiceDate"
    ]
]

,InvoiceNo,CustomerID,StockCode,Description,Quantity,UnitPrice,InvoiceDate
86789,543599,17560.0,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,0.0,2011-02-10 13:08:00
314748,564651,14646.0,21786,POLKADOT RAIN HAT,144,0.0,2011-08-26 14:19:00
314747,564651,14646.0,22955,36 FOIL STAR CAKE CASES,144,0.0,2011-08-26 14:19:00
314746,564651,14646.0,23268,SET OF 2 CERAMIC CHRISTMAS REINDEER,192,0.0,2011-08-26 14:19:00
314745,564651,14646.0,23270,SET OF 2 CERAMIC PAINTED HEARTS,96,0.0,2011-08-26 14:19:00
361825,568384,12748.0,M,Manual,1,0.0,2011-09-27 09:46:00
502122,578841,13256.0,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.0,2011-11-25 15:57:00


### Zero-price transaction treatment

Forty non-cancelled transaction lines have a UnitPrice of zero.

Thirty affected invoices also contain normally priced items, while four
invoices consist entirely of zero-price lines.

The zero-price-only invoices include unusually large quantities and a manual
transaction, indicating that these records do not represent standard
revenue-generating purchases.

For customer segmentation, core purchasing behaviour will therefore be based
only on transactions where `UnitPrice > 0`.

Zero-price transactions will be retained separately as non-revenue activity
so that they remain available for audit or potential behavioural analysis.
This prevents zero-value records from distorting purchase frequency, quantity,
average order value and monetary-value features.

In [34]:
# Preserve zero-price transactions separately rather than deleting them.
# These records do not contribute measurable revenue but may still be useful
# for audit or later behavioural analysis.

df_zero_price = df_purchases.loc[
    df_purchases["UnitPrice"] == 0
].copy()


# Create the core paid-purchase dataset used for customer purchase features.
# An invoice containing both paid and zero-price items remains represented
# because its positively priced lines are retained.

df_paid_purchases = df_purchases.loc[
    df_purchases["UnitPrice"] > 0
].copy()


print(f"Paid purchase transactions: {len(df_paid_purchases):,}")
print(f"Zero-price transactions: {len(df_zero_price):,}")

Paid purchase transactions: 392,692
Zero-price transactions: 40


In [35]:
# Confirm that separating zero-price activity has neither lost nor duplicated
# any of the non-cancelled transaction records.

assert (
    len(df_paid_purchases) + len(df_zero_price)
    == len(df_purchases)
)

print("Purchase reconciliation check passed.")

Purchase reconciliation check passed.


## 7. Product and Transaction Validity

The remaining paid-purchase dataset is reviewed for incomplete or unusual
product information before it is used for customer-level feature engineering.

The assessment focuses on:

- missing product descriptions;
- missing stock codes;
- zero or invalid quantities;
- non-standard stock codes that may represent administrative charges rather
  than merchandise.

Potentially unusual records are investigated before any exclusion rule is applied.

In [36]:
# Assess completeness of the product-identification fields used later for
# product diversity and purchasing-behaviour features.

product_missing_summary = pd.DataFrame({
    "missing_count": [
        df_paid_purchases["StockCode"].isna().sum(),
        df_paid_purchases["Description"].isna().sum()
    ],
    "missing_percentage": [
        round(df_paid_purchases["StockCode"].isna().mean() * 100, 2),
        round(df_paid_purchases["Description"].isna().mean() * 100, 2)
    ]
}, index=["StockCode", "Description"])

product_missing_summary

,missing_count,missing_percentage
StockCode,0,0.0
Description,0,0.0


### Finding

No missing values were identified in `StockCode` or `Description` within the
paid-purchase dataset.

No cleaning action is required for product-field completeness at this stage.

In [38]:
# Validate that all remaining paid-purchase records have positive quantities.

quantity_validation = pd.DataFrame({
    "Condition": [
        "Quantity < 0",
        "Quantity = 0",
        "Quantity > 0"
    ],
    "Count": [
        (df_paid_purchases["Quantity"] < 0).sum(),
        (df_paid_purchases["Quantity"] == 0).sum(),
        (df_paid_purchases["Quantity"] > 0).sum()
    ]
})

quantity_validation

,Condition,Count
0,Quantity < 0,0
1,Quantity = 0,0
2,Quantity > 0,392692


### Finding

All 392,692 transactions in the paid-purchase dataset have a positive quantity.

Negative quantities were previously isolated as cancelled/returned transactions,
and no zero-quantity records remain.

No further quantity-based cleaning is required for the core paid-purchase dataset.

In [42]:
# Identify stock codes that contain no numeric characters.
# These may represent administrative charges, manual adjustments,
# postage or other non-merchandise transaction types.

special_stock_mask = (
    ~df_paid_purchases["StockCode"]
    .astype("string")
    .str.contains(r"\d", na=False)
)

special_stock_rows = df_paid_purchases.loc[
    special_stock_mask
].copy()

print(
    f"Transactions with non-standard stock codes: "
    f"{len(special_stock_rows):,}"
)

Transactions with non-standard stock codes: 1,409


In [40]:
# Summarise non-standard stock codes to understand whether they represent
# genuine products or administrative/non-merchandise activity.

special_stock_summary = (
    special_stock_rows
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .agg(
        row_count=("InvoiceNo", "size"),
        unique_invoices=("InvoiceNo", "nunique"),
        unique_customers=("CustomerID", "nunique"),
        total_quantity=("Quantity", "sum")
    )
    .reset_index()
    .sort_values("row_count", ascending=False)
)

display(special_stock_summary)

,StockCode,Description,row_count,unique_invoices,unique_customers,total_quantity
4,POST,POSTAGE,1099,1099,331,3120
2,M,Manual,279,253,197,6933
1,DOT,DOTCOM POSTAGE,16,16,1,16
0,BANK CHARGES,Bank Charges,12,11,10,12
3,PADS,PADS TO MATCH ALL CUSHIONS,3,3,3,3


In [43]:
# Calculate transaction value so that the financial significance of each
# non-standard stock code can be assessed before determining its treatment.

special_stock_rows["LineValue"] = (
    special_stock_rows["Quantity"] *
    special_stock_rows["UnitPrice"]
)

special_stock_value_summary = (
    special_stock_rows
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .agg(
        row_count=("InvoiceNo", "size"),
        unique_customers=("CustomerID", "nunique"),
        min_unit_price=("UnitPrice", "min"),
        median_unit_price=("UnitPrice", "median"),
        max_unit_price=("UnitPrice", "max"),
        total_value=("LineValue", "sum")
    )
    .reset_index()
    .sort_values("total_value", ascending=False)
)

display(special_stock_value_summary)

,StockCode,Description,row_count,unique_customers,min_unit_price,median_unit_price,max_unit_price,total_value
4,POST,POSTAGE,1099,331,1.000,18.000,8142.750,77803.960
2,M,Manual,279,197,0.060,1.500,4161.060,53419.930
1,DOT,DOTCOM POSTAGE,16,1,11.170,715.850,1599.260,11906.360
0,BANK CHARGES,Bank Charges,12,10,0.001,15.000,15.000,165.001
3,PADS,PADS TO MATCH ALL CUSHIONS,3,3,0.001,0.001,0.001,0.003


In [45]:
# Review PADS separately because its description appears product-like,
# despite having a non-standard alphabetic StockCode.

display(
    df_paid_purchases.loc[
        df_paid_purchases["StockCode"] == "PADS",
        [
            "InvoiceNo",
            "CustomerID",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "InvoiceDate",
            "Country"
        ]
    ]
)

,InvoiceNo,CustomerID,StockCode,Description,Quantity,UnitPrice,InvoiceDate,Country
157195,550193,13952.0,PADS,PADS TO MATCH ALL CUSHIONS,1,0.001,2011-04-15 09:27:00,United Kingdom
279045,561226,15618.0,PADS,PADS TO MATCH ALL CUSHIONS,1,0.001,2011-07-26 10:13:00,United Kingdom
359871,568200,16198.0,PADS,PADS TO MATCH ALL CUSHIONS,1,0.001,2011-09-25 14:58:00,United Kingdom


### Treatment of non-standard StockCodes

Several non-standard StockCodes represent different types of customer or
administrative activity rather than standard merchandise.

The records will not be deleted. Instead, they will be classified by business
purpose so that different transaction types can be used appropriately during
feature engineering.

- `POST` and `DOT` represent postage and will be retained as potential
  fulfilment-related behavioural features.
- `M` represents manual transactions and will be retained separately from
  standard merchandise because its business meaning is not sufficiently clear
  to treat it as a product.
- `BANK CHARGES` represents administrative charges and will not be treated as
  merchandise.
- `PADS` has a nominal UnitPrice of £0.001 and will be retained as a
  non-standard item rather than contributing to core merchandise features.

All remaining StockCodes will be treated as standard merchandise.

In [47]:
# Map known non-standard StockCodes to meaningful business categories.
# All codes not listed here are treated as standard merchandise.

special_code_categories = {
    "POST": "Postage",
    "DOT": "Postage",
    "M": "Manual transaction",
    "BANK CHARGES": "Bank charge",
    "PADS": "Nominal-value item"
}

df_paid_purchases["TransactionCategory"] = (
    df_paid_purchases["StockCode"]
    .map(special_code_categories)
    .fillna("Merchandise")
)

# Review the number of transaction lines within each business category.

transaction_category_summary = (
    df_paid_purchases["TransactionCategory"]
    .value_counts()
    .rename_axis("TransactionCategory")
    .reset_index(name="row_count")
)

transaction_category_summary

,TransactionCategory,row_count
0,Merchandise,391283
1,Postage,1115
2,Manual transaction,279
3,Bank charge,12
4,Nominal-value item,3


In [48]:
# Calculate transaction-line value for later customer-level aggregation.

df_paid_purchases["LineValue"] = (
    df_paid_purchases["Quantity"] *
    df_paid_purchases["UnitPrice"]
)

# Summarise both transaction volume and monetary value by category.

category_value_summary = (
    df_paid_purchases
    .groupby("TransactionCategory")
    .agg(
        row_count=("InvoiceNo", "size"),
        unique_customers=("CustomerID", "nunique"),
        total_value=("LineValue", "sum")
    )
    .reset_index()
    .sort_values("total_value", ascending=False)
)

display(category_value_summary)

,TransactionCategory,row_count,unique_customers,total_value
2,Merchandise,391283,4334,8743913.640
4,Postage,1115,332,89710.320
1,Manual transaction,279,197,53419.930
0,Bank charge,12,10,165.001
3,Nominal-value item,3,3,0.003


In [49]:
# Standard products used to describe core purchasing behaviour.
df_merchandise = df_paid_purchases.loc[
    df_paid_purchases["TransactionCategory"] == "Merchandise"
].copy()

# Postage retained separately because it may describe fulfilment behaviour.
df_postage = df_paid_purchases.loc[
    df_paid_purchases["TransactionCategory"] == "Postage"
].copy()

# Other non-standard transactions retained for audit and possible
# exploratory behavioural features.
df_non_standard = df_paid_purchases.loc[
    df_paid_purchases["TransactionCategory"].isin(
        ["Manual transaction", "Bank charge", "Nominal-value item"]
    )
].copy()

## 8. Final Cleaning Validation and Reconciliation

The cleaned transaction datasets are validated before they are persisted for
feature engineering.

The checks confirm that:

- all core paid-purchase transactions belong to an identifiable customer;
- quantities and prices are positive;
- required transaction and product fields are populated;
- cancellations, zero-price activity and non-standard transaction types have
  been preserved separately rather than silently discarded;
- the movement from source records to cleaned datasets can be reconciled.

In [50]:
# Validate the core paid-purchase dataset before it is used for
# customer-level feature engineering.

validation_checks = {
    "Missing CustomerID": df_paid_purchases["CustomerID"].isna().sum(),
    "Missing InvoiceNo": df_paid_purchases["InvoiceNo"].isna().sum(),
    "Missing StockCode": df_paid_purchases["StockCode"].isna().sum(),
    "Missing Description": df_paid_purchases["Description"].isna().sum(),
    "Quantity <= 0": (df_paid_purchases["Quantity"] <= 0).sum(),
    "UnitPrice <= 0": (df_paid_purchases["UnitPrice"] <= 0).sum(),
    "Missing InvoiceDate": df_paid_purchases["InvoiceDate"].isna().sum(),
    "Missing Country": df_paid_purchases["Country"].isna().sum()
}

validation_summary = (
    pd.DataFrame(
        validation_checks.items(),
        columns=["Validation check", "Issue count"]
    )
)

validation_summary

,Validation check,Issue count
0,Missing CustomerID,0
1,Missing InvoiceNo,0
2,Missing StockCode,0
3,Missing Description,0
4,Quantity <= 0,0
5,UnitPrice <= 0,0
6,Missing InvoiceDate,0
7,Missing Country,0


In [51]:
# Confirm the temporal coverage of the cleaned purchase dataset.

print(
    "Earliest paid transaction:",
    df_paid_purchases["InvoiceDate"].min()
)

print(
    "Latest paid transaction:",
    df_paid_purchases["InvoiceDate"].max()
)

Earliest paid transaction: 2010-12-01 08:26:00
Latest paid transaction: 2011-12-09 12:50:00


In [52]:
# Reconcile the source dataset against each major cleaning decision.

cleaning_reconciliation = pd.DataFrame([
    {
        "Stage": "Original source dataset",
        "Rows": len(df),
        "Rows removed/excluded": 0,
        "Reason": "Registered source MLTable"
    },
    {
        "Stage": "After missing CustomerID removal",
        "Rows": len(
            df.loc[df["CustomerID"].notna()]
        ),
        "Rows removed/excluded": df["CustomerID"].isna().sum(),
        "Reason": "Customer cannot be reliably identified"
    },
    {
        "Stage": "After exact duplicate removal",
        "Rows": len(df_clean),
        "Rows removed/excluded": duplicates_to_remove,
        "Reason": "Probable duplicate transaction lines"
    },
    {
        "Stage": "Cancelled / return activity",
        "Rows": len(df_returns),
        "Rows removed/excluded": len(df_returns),
        "Reason": "Preserved separately from positive purchase behaviour"
    },
    {
        "Stage": "Zero-price activity",
        "Rows": len(df_zero_price),
        "Rows removed/excluded": len(df_zero_price),
        "Reason": "Preserved separately from revenue-generating purchases"
    },
    {
        "Stage": "Final paid-purchase dataset",
        "Rows": len(df_paid_purchases),
        "Rows removed/excluded": 0,
        "Reason": "Dataset used for customer purchase feature engineering"
    }
])

display(
    cleaning_reconciliation.style.set_properties(
        **{
            "white-space": "normal",
            "text-align": "left",
            "vertical-align": "top"
        }
    )
)

,Stage,Rows,Rows removed/excluded,Reason
0,Original source dataset,541909,0,Registered source MLTable
1,After missing CustomerID removal,406829,135080,Customer cannot be reliably identified
2,After exact duplicate removal,401604,5225,Probable duplicate transaction lines
3,Cancelled / return activity,8872,8872,Preserved separately from positive purchase behaviour
4,Zero-price activity,40,40,Preserved separately from revenue-generating purchases
5,Final paid-purchase dataset,392692,0,Dataset used for customer purchase feature engineering


In [53]:
# Summarise the datasets retained at the end of the cleaning process.

cleaned_dataset_summary = pd.DataFrame({
    "Dataset": [
        "Core paid purchases",
        "Returns / cancellations",
        "Zero-price activity",
        "Postage activity",
        "Other non-standard activity"
    ],
    "Rows": [
        len(df_paid_purchases),
        len(df_returns),
        len(df_zero_price),
        len(df_postage),
        len(df_non_standard)
    ]
})

cleaned_dataset_summary

,Dataset,Rows
0,Core paid purchases,392692
1,Returns / cancellations,8872
2,Zero-price activity,40
3,Postage activity,1115
4,Other non-standard activity,294


In [54]:
# Review the final columns that will be carried forward.

df_paid_purchases.info()

<class 'pandas.core.frame.DataFrame'>
Index: 392692 entries, 0 to 541908
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   InvoiceNo            392692 non-null  string        
 1   StockCode            392692 non-null  string        
 2   Description          392692 non-null  string        
 3   Quantity             392692 non-null  int64         
 4   InvoiceDate          392692 non-null  datetime64[ns]
 5   UnitPrice            392692 non-null  float64       
 6   CustomerID           392692 non-null  float64       
 7   Country              392692 non-null  string        
 8   TransactionCategory  392692 non-null  object        
 9   LineValue            392692 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1), string(4)
memory usage: 33.0+ MB


In [55]:
# Persist the cleaned paid-purchase dataset in Parquet format.
# Parquet preserves the schema and is efficient for repeated downstream processing.

cleaned_parquet_path = "../data/processed/online_retail_paid_purchases.parquet"

df_paid_purchases.to_parquet(
    cleaned_parquet_path,
    engine="pyarrow",
    index=False
)

print(f"Cleaned dataset saved to: {cleaned_parquet_path}")

Cleaned dataset saved to: ../data/processed/online_retail_paid_purchases.parquet


In [56]:
# Read the persisted Parquet file back into memory to confirm that the
# saved dataset matches the cleaned DataFrame.

df_paid_check = pd.read_parquet(cleaned_parquet_path)

print("DataFrame shape:", df_paid_purchases.shape)
print("Parquet shape:", df_paid_check.shape)

DataFrame shape: (392692, 10)
Parquet shape: (392692, 10)


In [57]:
# Define a versioned Azure ML Data Asset for the cleaned Parquet file.
# Because the path is local, Azure ML will upload the file to the workspace
# storage when the asset is registered.

paid_parquet_asset = Data(
    name="online-retail-paid-purchases-parquet",
    version="1",
    type=AssetTypes.URI_FILE,
    path=cleaned_parquet_path,
    description=(
        "Cleaned paid-purchase transactions for the Online Retail "
        "customer segmentation project."
    )
)

In [58]:
# Register the cleaned Parquet file in the Azure ML workspace.

registered_paid_parquet = ml_client.data.create_or_update(
    paid_parquet_asset
)

Uploading online_retail_paid_purchases.parquet (< 1 MB): 100%|██████████| 3.11M/3.11M [00:00<00:00, 45.3MB/s]




In [59]:
print("Name:", registered_paid_parquet.name)
print("Version:", registered_paid_parquet.version)
print("Type:", registered_paid_parquet.type)
print("Cloud path:", registered_paid_parquet.path)

Name: online-retail-paid-purchases-parquet
Version: 1
Type: uri_file
Cloud path: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/a4d70fae86766d751f41bfd05513fecbc2ca66cffdc231dbcd0c6d0927a283f6/online_retail_paid_purchases.parquet


In [60]:
import mltable

# Create an MLTable definition that references the registered
# cleaned Parquet file in Azure storage.

paid_purchase_table = mltable.from_parquet_files(
    paths=[
        {
            "file": registered_paid_parquet.path
        }
    ]
)

In [61]:
# Materialise the MLTable as a Pandas DataFrame to verify that Azure ML
# can successfully resolve and read the cleaned dataset.

paid_purchase_test = (
    paid_purchase_table
    .to_pandas_dataframe()
)

print("MLTable shape:", paid_purchase_test.shape)

paid_purchase_test.head()

MLTable shape: (392692, 10)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionCategory,LineValue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,Merchandise,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Merchandise,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,Merchandise,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Merchandise,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Merchandise,20.34


In [62]:
# Confirm that no records were lost when moving from the cleaned
# DataFrame to Parquet and then to MLTable.

assert paid_purchase_test.shape == df_paid_purchases.shape

print("MLTable validation passed.")

MLTable validation passed.


In [63]:
import os

paid_mltable_folder = "../data/mltable/paid-purchases"

os.makedirs(
    paid_mltable_folder,
    exist_ok=True
)

# save the MLTable
paid_purchase_table.save(paid_mltable_folder)

In [71]:
# Register the cleaned MLTable as the governed input for
# customer-level feature engineering.

paid_purchase_mltable_asset = Data(
    name="online-retail-paid-purchases",
    version="1",
    type=AssetTypes.MLTABLE,
    path=paid_mltable_folder,
    description=(
        "Cleaned and validated paid-purchase transactions used "
        "for customer-level feature engineering and clustering."
    )
)

# register the MLTable as a Data Asset
registered_paid_mltable = ml_client.data.create_or_update(
    paid_purchase_mltable_asset
)

In [73]:
print(registered_paid_mltable.name)
print(registered_paid_mltable.version)
print(registered_paid_mltable.type)

online-retail-paid-purchases
1
mltable


In [74]:
# Persist cancelled/returned transactions so that return behaviour can be
# engineered independently in the downstream feature engineering stage.

returns_parquet_path = "../data/processed/online_retail_returns.parquet"

df_returns.to_parquet(
    returns_parquet_path,
    engine="pyarrow",
    index=False
)

print(f"Returns dataset saved to: {returns_parquet_path}")

Returns dataset saved to: ../data/processed/online_retail_returns.parquet


In [75]:
# verfy

df_returns_check = pd.read_parquet(returns_parquet_path)

print("Original returns shape:", df_returns.shape)
print("Saved returns shape:", df_returns_check.shape)

assert df_returns_check.shape == df_returns.shape

print("Returns dataset validation passed.")

Original returns shape: (8872, 8)
Saved returns shape: (8872, 8)
Returns dataset validation passed.


In [76]:
# Register the persisted returns dataset as a versioned Azure ML data asset.

returns_parquet_asset = Data(
    name="online-retail-returns-parquet",
    version="1",
    type=AssetTypes.URI_FILE,
    path=returns_parquet_path,
    description=(
        "Cancelled and returned Online Retail transactions retained "
        "for customer return-behaviour feature engineering."
    )
)

registered_returns_parquet = ml_client.data.create_or_update(
    returns_parquet_asset
)

Uploading online_retail_returns.parquet (< 1 MB): 100%|██████████| 172k/172k [00:00<00:00, 7.37MB/s]




In [77]:
print("Name:", registered_returns_parquet.name)
print("Version:", registered_returns_parquet.version)
print("Type:", registered_returns_parquet.type)
print("Path:", registered_returns_parquet.path)

Name: online-retail-returns-parquet
Version: 1
Type: uri_file
Path: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/6639830e1e934f123d821ca5540c719c03106e1b01071d54d7d9cd36b39cb0ee/online_retail_returns.parquet


In [78]:
# Create an MLTable definition that points to the registered returns Parquet file.

returns_table = mltable.from_parquet_files(
    paths=[
        {
            "file": registered_returns_parquet.path
        }
    ]
)

In [79]:
# Materialise the MLTable to confirm that the returns data can be read correctly.

returns_test = returns_table.to_pandas_dataframe()

print("Returns MLTable shape:", returns_test.shape)

Returns MLTable shape: (8872, 8)


In [80]:
# Create a project folder for the returns MLTable definition.

returns_mltable_folder = "../data/mltable/returns"

os.makedirs(
    returns_mltable_folder,
    exist_ok=True
)

# Save the MLTable definition file.

returns_table.save(
    returns_mltable_folder
)

paths:
- file: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/6639830e1e934f123d821ca5540c719c03106e1b01071d54d7d9cd36b39cb0ee/online_retail_returns.parquet
transformations:
- read_parquet:
    include_path_column: false
    path_column: Path
type: mltable

In [81]:
# Register the returns MLTable as a versioned Azure ML data asset.

returns_mltable_asset = Data(
    name="online-retail-returns",
    version="1",
    type=AssetTypes.MLTABLE,
    path=returns_mltable_folder,
    description=(
        "Cancelled and returned Online Retail transactions used "
        "for customer return-behaviour feature engineering."
    )
)

# register a
registered_returns_mltable = ml_client.data.create_or_update(
    returns_mltable_asset
)

In [82]:
registered_returns_mltable = ml_client.data.create_or_update(
    returns_mltable_asset
)

Uploading returns (0.0 MBs): 100%|██████████| 385/385 [00:00<00:00, 17078.69it/s]


